# Referential Integrity Execution
Inspect controlled SQL, execute approved/automatic relationships, and write table-level and consolidated reports. Review candidates are never executed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import TablePair, load_app_config, load_business_context, load_table_pairs
from dq_agent.connectors import BigQueryConnector
from dq_agent.context_utils import configure_workflow_logging, workflow_paths, logged_step
from dq_agent.query_engine import QueryGuard, SQLCompiler, allowed_tables_for_rule
from dq_agent.relationships import relationship_rule
from dq_agent.reporting import write_relationship_reports

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DISCOVERY_RUN_ID = None  # None selects the newest relationship discovery output
USE_SAMPLE_RESULTS = True
EXECUTE_LIVE_CHECKS = False
config = load_app_config(ROOT)
paths = workflow_paths(config, RUN_ID, 'relationships')
logger = configure_workflow_logging(paths['log'], config.project.log_level)
business_context = load_business_context(config)
guard = QueryGuard()

In [ ]:
with logged_step(logger, paths['checkpoint'], 'LOAD_RELATIONSHIP_CONFIGURATION'):
    relationship_root = config.path(config.project.outputs_dir) / 'relationships'
    if DISCOVERY_RUN_ID is None:
        discovery_files = sorted(relationship_root.glob('*/relationship_candidates.json'), key=lambda path: path.stat().st_mtime)
        if not discovery_files:
            raise FileNotFoundError('Run 04_relationship_discovery.ipynb first')
        candidate_file = discovery_files[-1]
    else:
        candidate_file = relationship_root / DISCOVERY_RUN_ID / 'relationship_candidates.json'
    candidates = json.loads(candidate_file.read_text(encoding='utf-8'))
    selected = [item for item in candidates if item.get('decision') == 'AUTO']
    logger.info('LOAD_RELATIONSHIP_CONFIGURATION candidate_file=%s total=%s executable=%s', candidate_file, len(candidates), len(selected))
display(pd.DataFrame(selected)[['relationship_id','pair_id','child_columns','parent_table','parent_columns','origin','confidence','decision']])

In [ ]:
if USE_SAMPLE_RESULTS:
    pairs = [
        TablePair(pair_id='sample_fact_sales', mode='bigquery_only', target_project='your-gcp-project', target_dataset='analytics', target_table='fact_sales', context_id='sample_fact_sales'),
        TablePair(pair_id='sample_dim_market', mode='bigquery_only', target_project='your-gcp-project', target_dataset='analytics', target_table='dim_market', context_id='sample_dim_market'),
    ]
else:
    pairs = load_table_pairs(config)
pair_by_id = {pair.pair_id: pair for pair in pairs}

In [ ]:
with logged_step(logger, paths['checkpoint'], 'GENERATE_REFERENTIAL_INTEGRITY_SQL'):
    sql_rows = []
    sql_dir = paths['output'] / 'sql'
    sql_dir.mkdir(parents=True, exist_ok=True)
    for candidate in selected:
        pair = pair_by_id[candidate['pair_id']]
        table_context = business_context.get('tables', {}).get(pair.context_id or pair.pair_id, {})
        base_filters = list(table_context.get('filters', {}).get('target', []))
        rule = relationship_rule(candidate)
        sql = SQLCompiler('bigquery').compile_rule(rule, pair, 'target', base_filters)
        checked_sql = guard.validate(sql, 'bigquery', allowed_tables_for_rule(rule, pair, 'target'))
        sql_file = sql_dir / f"{candidate['candidate_id']}.sql"
        sql_file.write_text(checked_sql + '\n', encoding='utf-8')
        candidate['validated_sql'] = checked_sql
        candidate['sql_file'] = str(sql_file)
        sql_rows.append({'candidate_id': candidate['candidate_id'], 'relationship_id': candidate['relationship_id'], 'sql': checked_sql, 'sql_file': str(sql_file)})
        logger.info('GENERATE_REFERENTIAL_INTEGRITY_SQL relationship=%s origin=%s filters=%s output=%s', candidate['relationship_id'], candidate['origin'], {'child': candidate.get('child_filters', []), 'parent': candidate.get('parent_filters', [])}, sql_file)
display(pd.DataFrame(sql_rows))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'EXECUTE_REFERENTIAL_INTEGRITY_CHECK'):
    results = []
    connector = None
    if EXECUTE_LIVE_CHECKS:
        if not pairs:
            raise ValueError('No enabled target tables')
        connector = BigQueryConnector(config.project.query_limits, pairs[0].target_project)
    for candidate in selected:
        try:
            if EXECUTE_LIVE_CHECKS:
                evidence = connector.execute(candidate['validated_sql']).frame.iloc[0].to_dict()
            elif USE_SAMPLE_RESULTS:
                evidence = candidate.get('loaded_relationship_profile') or candidate.get('relationship_profile') or {}
            else:
                raise ValueError('Set EXECUTE_LIVE_CHECKS=True or USE_SAMPLE_RESULTS=True')
            checked = int(evidence.get('checked_count') or 0)
            orphan = int(evidence.get('orphan_count') or 0)
            matched = int(evidence.get('matched_count') or checked - orphan)
            result = {
                'pair_id': candidate['pair_id'], 'relationship_id': candidate['relationship_id'],
                'candidate_id': candidate['candidate_id'], 'status': 'PASS' if orphan == 0 else 'FAIL',
                'checked_count': checked, 'matched_count': matched, 'orphan_count': orphan,
                'origin': candidate['origin'], 'confidence': candidate['confidence'],
                'child_table': candidate['child_table'], 'child_columns': candidate['child_columns'],
                'parent_table': candidate['parent_table'], 'parent_columns': candidate['parent_columns'],
                'child_filters': candidate.get('child_filters', []), 'parent_filters': candidate.get('parent_filters', []),
                'sql_file': candidate['sql_file'],
            }
        except Exception as exc:
            logger.exception('EXECUTE_REFERENTIAL_INTEGRITY_CHECK failed relationship=%s', candidate['relationship_id'])
            result = {'pair_id': candidate['pair_id'], 'relationship_id': candidate['relationship_id'], 'candidate_id': candidate['candidate_id'], 'status': 'ERROR', 'error': str(exc)}
        results.append(result)
        logger.info('EXECUTE_REFERENTIAL_INTEGRITY_CHECK relationship=%s status=%s checked=%s failed=%s', result['relationship_id'], result['status'], result.get('checked_count'), result.get('orphan_count'))
    if connector:
        connector.close()
display(pd.DataFrame(results))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'WRITE_RELATIONSHIP_RESULTS'):
    report_paths = write_relationship_reports(paths['output'], candidates, results)
    logger.info('WRITE_RELATIONSHIP_RESULTS candidates=%s results=%s outputs=%s', len(candidates), len(results), report_paths)
display(report_paths)

Candidates marked `REVIEW` are excluded. Import their workbook with notebook `02_approval_processing.ipynb`, rerun discovery, and verify they return as trusted relationships before live execution.